# Assignment 4 — Live Demo 


## 1) CONFIG


In [1]:
from pathlib import Path
import json, random
from IPython.display import display, Audio, Markdown, HTML

ROOT = Path("/home/administrator/assignmen-kzyn/KyawZayaNaing-assignment4")
DATA = ROOT / "kaldi_recipe" / "data"
OUT  = ROOT / "kaldi_data"
EXP  = ROOT / "kaldi_recipe" / "exp"
PLAYLIST_PATH = ROOT / "demo" / "playlist.json"

# ─── Override (None = use playlist.json) ───
OVERRIDE_MODEL = None          # e.g. "hf_w2v2" | "ctc" | "dnn" | "mono" | "tri1"
OVERRIDE_MODE  = None          # "playlist" | "speakers" | "custom_ids" | "compare_models"
OVERRIDE_UTTS  = None          # e.g. ["PunKhattgar-Rec1-0001-..."]
OVERRIDE_SPEAKERS = None       # e.g. ["YeLinTun"]
SHOW_COMPARE_MODELS = False  # True = same utts across all models that have decode

cfg = json.loads(PLAYLIST_PATH.read_text(encoding="utf-8"))
MODEL = OVERRIDE_MODEL or cfg.get("model", "hf_w2v2")
MODE  = OVERRIDE_MODE or cfg.get("mode", "playlist")
SPEAKERS = OVERRIDE_SPEAKERS or cfg.get("speakers", ["YeLinTun", "PunKhattgar"])
N_PER = int(cfg.get("n_per_speaker", 3))
PREFER = cfg.get("prefer", "best")  # best | worst | random
SHOW_AUDIO = bool(cfg.get("show_audio", True))

print("playlist file:", PLAYLIST_PATH)
print("MODEL =", MODEL, "| MODE =", MODE, "| SPEAKERS =", SPEAKERS)


playlist file: /home/administrator/assignmen-kzyn/KyawZayaNaing-assignment4/demo/playlist.json
MODEL = hf_w2v2 | MODE = playlist | SPEAKERS = ['YeLinTun', 'PunKhattgar']


## 2) Load REF / Result / wav paths


In [2]:
def load_text(path: Path) -> dict[str, str]:
    out = {}
    if not path.exists():
        return out
    for ln in path.read_text(encoding="utf-8").splitlines():
        parts = ln.split(maxsplit=1)
        if len(parts) == 2:
            out[parts[0]] = parts[1]
        elif len(parts) == 1:
            out[parts[0]] = ""
    return out

def load_wav_scp(path: Path) -> dict[str, Path]:
    out = {}
    if not path.exists():
        return out
    for ln in path.read_text().splitlines():
        if not ln.strip():
            continue
        utt, wav = ln.split(maxsplit=1)
        out[utt] = Path(wav)
    return out

def char_wer(ref: str, hyp: str) -> float:
    r = list(ref.replace(" ", ""))
    h = list(hyp.replace(" ", ""))
    n, m = len(r), len(h)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if r[i - 1] == h[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[n][m] / max(n, 1)

ref = load_text(DATA / "test" / "text_orig") or load_text(OUT / "test" / "text")
wavs = load_wav_scp(OUT / "test" / "wav.scp") or load_wav_scp(DATA / "test" / "wav.scp")
u2s_path = OUT / "test" / "utt2spk"
if not u2s_path.exists():
    u2s_path = DATA / "test" / "utt2spk"
u2s = dict(ln.split() for ln in u2s_path.read_text().splitlines() if ln.strip())

available = [m for m in ("hf_w2v2", "ctc", "dnn", "mono", "tri1")
             if (EXP / m / "decode_test" / "trans.txt").exists()]
print("Available decode models:", available)
if MODEL not in available:
    MODEL = available[0] if available else None
    print("→ falling back to", MODEL)

hyp = load_text(EXP / MODEL / "decode_test" / "trans.txt") if MODEL else {}
print(f"Using MODEL={MODEL}  hyp={len(hyp)}  ref={len(ref)}  wavs={len(wavs)}")


Available decode models: ['hf_w2v2', 'ctc', 'dnn', 'mono', 'tri1']
Using MODEL=hf_w2v2  hyp=1420  ref=1420  wavs=1420


## 3) Build show-list (playlist / speakers / custom IDs)


In [3]:
notes = {}
show = []

if OVERRIDE_UTTS:
    show = list(OVERRIDE_UTTS)
    MODE = "custom_ids"
elif MODE == "playlist":
    for item in cfg.get("playlist", []):
        if isinstance(item, str):
            show.append(item)
        else:
            u = item.get("utt")
            if u:
                show.append(u)
                if item.get("note"):
                    notes[u] = item["note"]
elif MODE == "custom_ids":
    show = list(cfg.get("utt_ids") or [])
elif MODE == "speakers":
    common = [u for u in hyp if u in ref and u in wavs]
    by_spk = {}
    for u in common:
        by_spk.setdefault(u2s.get(u, "?"), []).append(u)
    for spk in SPEAKERS:
        utts = by_spk.get(spk, [])
        scored = [(char_wer(ref[u], hyp.get(u, "")), u) for u in utts]
        if PREFER == "worst":
            scored.sort(reverse=True)
        elif PREFER == "random":
            random.shuffle(scored)
        else:
            scored.sort()
        show.extend([u for _, u in scored[:N_PER]])
else:
    show = [item["utt"] if isinstance(item, dict) else item for item in cfg.get("playlist", [])]

# keep only existing
show = [u for u in show if u in ref and u in wavs]
missing_hyp = [u for u in show if u not in hyp]
print(f"Will show {len(show)} utterances (MODE={MODE})")
if missing_hyp:
    print("WARNING: no HYP for", len(missing_hyp), "utts (model not decoded?):", missing_hyp[:3])
for i, u in enumerate(show, 1):
    print(f"  {i}. [{u2s.get(u,'?')}] {u}")


Will show 4 utterances (MODE=playlist)
  1. [YeLinTun] YeLinTun-Rec1-0011-yelintun_20260915_215648
  2. [PunKhattgar] PunKhattgar-Rec1-0014-PunKhattgar_20260914_125058
  3. [PunKhattgar] PunKhattgar-Rec1-0001-PunKhattgar_20260914_124421
  4. [YeLinTun] YeLinTun-Rec5-0057-yelintun_20260916_180531


## 4) Play demo


In [4]:
def render_one(utt: str, model_name: str, result_map: dict[str, str]):
    r = ref.get(utt, "")
    h = result_map.get(utt, "")
    wer = char_wer(r, h) if utt in result_map else None
    ok = (wer == 0.0) if wer is not None else False
    badge = "✅ MATCH" if ok else ("❌ ERROR" if wer is not None else "— no result")
    color = "#0a7" if ok else "#c33"
    note = notes.get(utt, "")
    display(HTML(f"""
    <div style='border:1px solid #ddd;border-radius:10px;padding:12px 14px;margin:10px 0;
                font-family:system-ui,sans-serif'>
      <div style='font-size:13px;color:#666'>Model: <b>{model_name}</b> · Speaker: <b>{u2s.get(utt,'?')}</b></div>
      <div style='font-size:12px;color:#888;word-break:break-all'>{utt}</div>
      {f"<div style='margin-top:4px;color:#555'><i>{note}</i></div>" if note else ""}
      <div style='margin-top:8px;font-size:20px'><b>REF:</b> {r if r else '<i>(empty)</i>'}</div>
      <div style='font-size:20px'><b>Result:</b> {h if h else '<i>(empty)</i>'}</div>
      <div style='margin-top:6px;color:{color};font-weight:600'>
        {badge}{f" · Char WER={wer*100:.0f}%" if wer is not None else ""}
      </div>
    </div>
    """))
    if SHOW_AUDIO:
        w = wavs.get(utt)
        if w and w.exists():
            display(Audio(filename=str(w)))
        else:
            print("wav missing:", w)

if not show:
    display(Markdown("**No utterances to show.** Edit `demo/playlist.json` or set `OVERRIDE_UTTS`."))
elif SHOW_COMPARE_MODELS or MODE == "compare_models":
    for utt in show:
        display(Markdown(f"### UTT `{utt}`"))
        for m in available:
            hm = load_text(EXP / m / "decode_test" / "trans.txt")
            render_one(utt, m, hm)
else:
    display(Markdown(f"## Demo model: `{MODEL}` — {len(show)} clips"))
    for utt in show:
        render_one(utt, MODEL, hyp)


## Demo model: `hf_w2v2` — 4 clips

## 5) Interactive picker


In [5]:
# Interactive audio picker (ipywidgets)
try:
    import ipywidgets as W
    from IPython.display import clear_output
except ImportError as e:
    raise SystemExit(
        "ipywidgets မရှိပါ — `pip install ipywidgets` လုပ်ပါ။\n"
        "သို့မဟုတ် အောက်က fallback cell မှာ UTT_ID ရိုက်သုံးပါ။"
    ) from e

# Build speaker → utt list (only those with wav + ref)
_by_spk: dict[str, list[str]] = {}
for _u, _spk in u2s.items():
    if _u in ref and _u in wavs:
        _by_spk.setdefault(_spk, []).append(_u)
for _spk in _by_spk:
    _by_spk[_spk].sort()
_speakers = sorted(_by_spk.keys())
if not _speakers:
    raise SystemExit("No test utterances with wav+ref found.")

_models = available[:] if available else ["hf_w2v2"]
_default_model = MODEL if MODEL in _models else _models[0]

dd_model = W.Dropdown(options=_models, value=_default_model, description="Model:", layout=W.Layout(width="420px"))
dd_spk = W.Dropdown(options=_speakers, value=_speakers[0], description="Speaker:", layout=W.Layout(width="420px"))
dd_utt = W.Dropdown(options=_by_spk[_speakers[0]], description="Audio:", layout=W.Layout(width="700px"))
chk_compare = W.Checkbox(value=False, description="Compare all models on this clip")
btn = W.Button(description="▶ Show / Play", button_style="success", layout=W.Layout(width="200px"))
out = W.Output()

def _refresh_utts(*_):
    spk = dd_spk.value
    opts = _by_spk.get(spk, [])
    dd_utt.options = opts if opts else ["(no match)"]
    if opts:
        dd_utt.value = opts[0]

try:
    dd_spk.unobserve_all()
except Exception:
    pass
dd_spk.observe(_refresh_utts, names="value")

def _on_show(_):
    # Always clear previous result before rendering a new one
    out.clear_output(wait=True)

    utt = dd_utt.value
    if not utt or utt == "(no match)" or utt not in ref:
        with out:
            print("Utterance မရွေးရသေး / မတွေ့ပါ။ Speaker ပြန်ရွေးပါ။")
        return

    model_name = dd_model.value

    with out:
        hm = load_text(EXP / model_name / "decode_test" / "trans.txt")

        # Render only one result
        print(f"Selected: {utt}")
        print(f"wav: {wavs.get(utt)}")
        render_one(utt, model_name, hm)

# Remove old callbacks when notebook cell is re-run
try:
    btn._click_handlers.callbacks = []
except Exception:
    pass

btn.on_click(_on_show)

display(Markdown("### Pick any test audio"))
display(W.VBox([dd_model, dd_spk, dd_utt, chk_compare, btn, out]))
print(f"Speakers: {len(_speakers)} · clips with wav+ref: {sum(len(v) for v in _by_spk.values())}")


### Pick any test audio

Speakers: 2 · clips with wav+ref: 1420


In [6]:

UTT_ID = "yelintun_20260916_152122"  

def resolve_utt(qid: str) -> str | None:
    q = (qid or "").strip()
    if not q:
        return None
    if q in ref:
        return q
    q_cf = q.casefold()
    hits = []
    for u in ref:
        u_cf = u.casefold()
        if q_cf in u_cf or u_cf.endswith(q_cf):
            hits.append(u)
            continue
        # bare timestamp e.g. 152122
        if q.isdigit() and q in u:
            hits.append(u)
    # unique preserve order
    seen, uniq = set(), []
    for h in hits:
        if h not in seen:
            seen.add(h)
            uniq.append(h)
    hits = uniq
    if len(hits) == 1:
        return hits[0]
    if hits:
        print(f"Multiple matches for {q!r}:")
        for h in hits[:20]:
            print(" ", h)
        return None
    return None

resolved = resolve_utt(UTT_ID)
if resolved is None:
    print("Unknown UTT_ID. Examples:")
    for u in list(ref)[:8]:
        print(" ", u)
else:
    if resolved != UTT_ID:
        print("Resolved →", resolved)
    render_one(resolved, MODEL or "?", hyp)


Resolved → YeLinTun-Rec1-0131-yelintun_20260916_152122


In [7]:
model_name = "hf_w2v2"

hm = load_text(
    EXP / model_name / "decode_test" / "trans.txt"
)

print("Loaded:", len(hm))

Loaded: 1420


In [8]:
utt = "PunKhattgar-Rec1-0124-PunKhattgar_20260914_133630"

print("REF:", ref.get(utt))

print("Result:", hm.get(utt))

REF: အရေအတွက် ၂၃ ခုပါ
Result: အရေအတွက် ၅၀၉ ခုပါ


In [9]:
utt = dd_utt.value

ref_text = ref.get(utt, "")
result_text = hm.get(utt, "")

print("UTT:", utt)

print("\nREF length:", len(ref_text))
print("REF:", ref_text)

print("\nResult length:", len(result_text))
print("Result:", result_text)

UTT: PunKhattgar-Rec1-0001-PunKhattgar_20260914_124421

REF length: 1
REF: ၀

Result length: 1
Result: ၀
